In [1]:
import os
from collections import defaultdict
from itertools import chain

import polars as pl

import src.social_groups.polars_columns as plc
from social_groups.analysis.defs.notebooks.definitions import register_materialization
from social_groups.analysis.polars_transformations import make_group_constellation
from social_groups.analysis.polars_transformations.apply_parsing_and_group_decision import (
    apply_parsing_and_group_decision,
)
from social_groups.directories import REPORTING_DIR
from social_groups.polars_values import MODEL_NAME_TO_LETTER_MAPPING
from social_groups.reporting.analysis_columns import AnalysisColumn
from social_groups.reporting.group_decision_scheme import calculate_decision_scheme
from social_groups.reporting.group_reply import (
    GroupReplyAggregator,
    MajorityVote,
    SingularityVote,
)
from social_groups.reporting.parsing import (
    AnswerComparer,
    AnswerOptions,
    AnswerParser,
)

%load_ext autoreload
%autoreload 2

/Users/philipp/Documents/Studium/Informatik/Masterthesis/Repository/.venv/lib/python3.11/site-packages/dagstermill/manager.py:398: BetaWarning: Class `Manager` is currently in beta, and may have breaking changes in minor version releases, with behavior changes in patch releases.
  MANAGER_FOR_NOTEBOOK_INSTANCE = Manager()
/Users/philipp/Documents/Studium/Informatik/Masterthesis/Repository/src/social_groups/analysis/notebook_assets.py:138: BetaWarning: Class `LocalFileCodeReference` is currently in beta, and may have breaking changes in minor version releases, with behavior changes in patch releases.
  dg.LocalFileCodeReference(
/Users/philipp/Documents/Studium/Informatik/Masterthesis/Repository/src/social_groups/analysis/notebook_assets.py:143: BetaWarning: Class `LocalFileCodeReference` is currently in beta, and may have breaking changes in minor version releases, with behavior changes in patch releases.
  dg.LocalFileCodeReference(
/Users/philipp/Documents/Studium/Informatik/Masterth

In [2]:
output_dir = REPORTING_DIR / "heterogeneous_group"
os.makedirs(output_dir, exist_ok=True)

triple_underscore_handling = "wrong"

In [3]:
parser = AnswerParser(AnswerOptions.letters_A_to_J)
group_reply = GroupReplyAggregator(MajorityVote())
comparer = AnswerComparer(
    AnswerOptions.letters_A_to_J, triple_underscore_handling=triple_underscore_handling
)

## Analzying Basic Group Behaviour

In [4]:
from social_groups.analysis.definitions import defs

mad_frame = defs().load_asset_value("hetero_mad")
baseline_frame = defs().load_asset_value("baseline")
original_table_page46 = defs().load_asset_value(
    ["report", "external", "group_problem_solving_page_46"]
)

/Users/philipp/Documents/Studium/Informatik/Masterthesis/Repository/.venv/lib/python3.11/site-packages/dagster/_config/pythonic_config/typing_utils.py:101: UserWarning: Field name "extension" in "PolarsParquetIOManager" shadows an attribute in parent "BasePolarsUPathIOManager"
  return super().__new__(cls, name, bases, namespaces, **kwargs)
2026-04-07 13:35:29 +0800 - dagster - DEBUG - system - Loading file from: /Users/philipp/Documents/Studium/Informatik/Masterthesis/Repository/results/analysis/dagster/hetero_mad.parquet using PolarsParquetIOManager...
2026-04-07 13:35:29 +0800 - dagster - DEBUG - system - Loading file from: /Users/philipp/Documents/Studium/Informatik/Masterthesis/Repository/results/analysis/dagster/baseline.parquet using PolarsParquetIOManager...
2026-04-07 13:35:30 +0800 - dagster - DEBUG - system - Loading file from: /Users/philipp/Documents/Studium/Informatik/Masterthesis/Repository/results/analysis/dagster/report/external/group_problem_solving_page_46 using Pick

In [ ]:
mad_frame.head()

In [6]:
# TODO: Is this mixed for different datasets? (MAD-> subset, Baseline -> Full Eval?)
baseline_analysis = (
    baseline_frame.with_columns(
        parser(pl.col("final_answer")).alias(AnalysisColumn.parsed_answer.value)
    )
    .with_columns(
        is_correct=comparer(
            pl.col(AnalysisColumn.parsed_answer.value), pl.col("answer_string")
        )
    )
    .group_by("model_name")
    .agg(accuracy=pl.col("is_correct").mean())
)

In [7]:
model_name_sort = {
    "Qwen/Qwen3-14B": 1,
    "Qwen/Qwen3-4B": 2,
    "Qwen/Qwen3-0.6B": 3,
}

mad_analysis = apply_parsing_and_group_decision(
    mad_frame, parser, comparer, group_reply
).with_columns(make_group_constellation())

table_page_46 = (
    pl.concat(
        [
            (
                mad_analysis.group_by("group_constellation")
                .agg(accuracy=pl.col("is_correct").mean())
                .with_columns(origin=pl.lit("(MAD)"))
            ),
            (
                baseline_analysis.with_columns(
                    group_constellation=pl.col("model_name").replace(
                        MODEL_NAME_TO_LETTER_MAPPING
                    ),
                ).with_columns(origin=pl.lit("(baseline)"))
            ),
        ],
        how="diagonal",
    )
    .drop("model_name")
    .sort(plc.accuracy, plc.group_constellation, "origin", descending=True)
)

register_materialization(
    "heterogeneous_groups_evaluation_table",
    table_page_46,
    "The main evaluation of different group constellations for Multi Agent Debate, can be nicely compared to the table on page 46 of Group Problem Solving Book.",
)
table_page_46

Skipping Materialization because in interactive mode.


/Users/philipp/Documents/Studium/Informatik/Masterthesis/Repository/.venv/lib/python3.11/site-packages/dagster/_core/execution/context_creation_job.py:273: RuntimeWarning: coroutine 'BaseEventLoop.shutdown_asyncgens' was never awaited
  pass


group_constellation,accuracy,origin
str,f64,str
"""HH""",0.76,"""(MAD)"""
"""HHH""",0.73,"""(MAD)"""
"""H""",0.72,"""(MAD)"""
"""HHM""",0.7,"""(MAD)"""
"""HHLM""",0.69,"""(MAD)"""
…,…,…
"""LLL""",0.41,"""(MAD)"""
"""L""",0.315,"""(baseline)"""
"""LL""",0.31,"""(MAD)"""


In [8]:
original_table_page46

In [9]:
decision_schemes = (
    mad_analysis.group_by("group_constellation")
    .map_groups(
        lambda g: calculate_decision_scheme(
            g,
            AnalysisColumn.parsed_individual_answers_before.value,
            AnalysisColumn.parsed_individual_answers_after.value,
            "answer_string",
            group_reply,
            comparer,
        ).select(
            pl.lit(g["group_constellation"].unique().item()).alias(
                "group_constellation"
            ),
            "Correct Members Beginning",
            "correct",
            "incorrect",
        )
    )
    .sort("group_constellation")
)

decision_schemes.write_csv(output_dir / "group_decision_schemes.csv")

decision_schemes

group_constellation,Correct Members Beginning,correct,incorrect
str,u32,f64,f64
"""H""",1,0.944444,0.055556
"""H""",0,0.142857,0.857143
"""HH""",2,0.983871,0.016129
"""HH""",1,0.6875,0.3125
"""HH""",0,0.181818,0.818182
…,…,…,…
"""MMMM""",4,0.925926,0.074074
"""MMMM""",3,0.666667,0.333333
"""MMMM""",2,0.666667,0.333333


## Unparsable answers:

In [10]:
unparsable_answers_per_model = defaultdict(int)

### In the baseline:

In [11]:
for answer in (
    baseline_frame.with_columns(
        parser(pl.col("final_answer")).alias(AnalysisColumn.parsed_answer.value)
    )
    .filter(pl.col(AnalysisColumn.parsed_answer.value).str.starts_with("___"))
    .select(["final_answer", "model_name"])
    .iter_rows()
):
    print(answer[1], ": \n")
    print(answer[0][-150:])
    print("-" * 50)
    unparsable_answers_per_model[answer[1]] += 1

Qwen/Qwen3-14B : 

s dose of DTaP or to any of its components**. Since that is not listed, the **best available option** is:

**(D): allergy to eggs** — as it is **not a
--------------------------------------------------
Qwen/Qwen3-14B : 

ext{K} $ again.

Try $ x = 150 \, \text{K} $ again.

Try $ x = 150 \, \text{K} $ again.

Try $ x = 150 \, \text{K} $ again.

Try $ x = 150 \, \text{K}
--------------------------------------------------
Qwen/Qwen3-14B : 

o large.

Let’s try **Option (F)** again:

- $ \Delta H_{\text{fus}} = 1.8 \, \text{kcal} = 7530 \, \text{J} $
- $ P_1 = 1.98 \, \text{mm Hg} = \frac{
--------------------------------------------------
Qwen/Qwen3-14B : 

comes a problem is **5.5 mm**, and the **minimum diameter** is **5.5 mm**.

So, the **maximum diameter** before diffraction becomes a problem is **5.5
--------------------------------------------------
Qwen/Qwen3-0.6B : 

0.70^{11} \cdot 4.6
$$

$$
P(\text{at least 3 women}) = 1 - 0.70^{11} \cdot 4.6
$$

$$
P(\text{a

## In the MAD:

In [12]:
for answer in chain(
    mad_analysis.filter(
        pl.col(AnalysisColumn.parsed_individual_answers_before.value)
        .list.eval(pl.element().str.starts_with("___"))
        .list.any()
    )
    .select(
        "answers_at_beginning",
        AnalysisColumn.parsed_individual_answers_before.value,
        "model_names",
    )
    .iter_rows(),
    mad_analysis.filter(
        pl.col(AnalysisColumn.parsed_individual_answers_before.value)
        .list.eval(pl.element().str.starts_with("___"))
        .list.any()
    )
    .select(
        "answers_at_end",
        AnalysisColumn.parsed_individual_answers_after.value,
        "model_names",
    )
    .iter_rows(),
):
    for i, parsed in enumerate(answer[1]):
        if parsed.startswith("___"):
            unparsable_answers_per_model[answer[2][i]] += 1
            print(parsed, f"from {answer[2][i]}: \n")
            print(answer[0][i][-150:])
            print("-" * 50)

___not_parsable___ from Qwen/Qwen3-0.6B: 

 h), where h is the head in meters. Let's try with h = 0.1 m. 

V = 0.95 * 0.92 * sqrt(2 * 9.81 * 0.1) ≈ 0.95 * 0.92 * sqrt(1.962) ≈ 0.95 * 0.92 * 1.4
--------------------------------------------------
___not_parsable___ from Qwen/Qwen3-0.6B: 

options again. Wait, maybe the total amount paid is $1840, and the down payment is $50, so the interest is $1790. Then, the interest rate is (1790 / 1
--------------------------------------------------
___not_parsable___ from Qwen/Qwen3-0.6B: 

hy, and a grade 3/6 murmur, all of which are characteristic of aortic dissection. The other options do not align with the clinical features described.
--------------------------------------------------
___not_parsable___ from Qwen/Qwen3-0.6B: 

l was $6 million, so the total is $106 million. The price increased by 20%, so the new price is $106 * 1.2 = $127.2 million. But the builder said they
--------------------------------------------------
___not_parsable___ f

-> Qwen 0.6B often says "Answer should be A /think. The Answer is G"

In [13]:
unparsable_answers_per_model

defaultdict(int,
            {'Qwen/Qwen3-14B': 330,
             'Qwen/Qwen3-0.6B': 908,
             'Qwen/Qwen3-4B': 385})

### Build an overview of parsing error Influence

In [14]:
baseline_frame_with_group_constellation = baseline_frame.with_columns(
    group_constellation=pl.col("model_name").replace(MODEL_NAME_TO_LETTER_MAPPING)
    + pl.lit(" (Baseline)")
).drop("model_name")

parsing_error_influence_table = pl.DataFrame(
    {
        "group_constellation": mad_analysis["group_constellation"]
        .unique()
        .extend(baseline_frame_with_group_constellation["group_constellation"].unique())
        .sort()
    }
)

for handling in ["null", "wrong", "random"]:
    new_comparer = AnswerComparer(AnswerOptions.letters_A_to_J, handling)

    new_mad = (
        apply_parsing_and_group_decision(mad_frame, parser, new_comparer, group_reply)
        .with_columns(make_group_constellation())
        .group_by("group_constellation")
        .agg(accuracy=pl.col("is_correct").mean())
    )

    new_base = (
        baseline_frame_with_group_constellation.with_columns(
            parser(pl.col("final_answer")).alias(AnalysisColumn.parsed_answer.value)
        )
        .with_columns(
            is_correct=new_comparer(
                pl.col(AnalysisColumn.parsed_answer.value), pl.col("answer_string")
            )
        )
        .group_by("group_constellation")
        .agg(accuracy=pl.col("is_correct").mean())
    )

    parsing_error_influence_table = parsing_error_influence_table.join(
        pl.concat([new_mad, new_base], how="diagonal").select(
            "group_constellation",
            pl.col("accuracy").alias(f"Accuracy ({handling})"),
        ),
        on="group_constellation",
        how="inner",
    )

parsing_error_influence_table.with_columns(
    deviation=(
        pl.max_horizontal(pl.exclude("group_constellation"))
        - pl.min_horizontal(pl.exclude("group_constellation"))
    ).alias("range")
)

group_constellation,Accuracy (null),Accuracy (wrong),Accuracy (random),deviation
str,f64,f64,f64,f64
"""HHL""",0.663265,0.65,0.65,0.013265
"""HHHL""",0.659794,0.64,0.64,0.019794
"""HLLM""",0.610526,0.58,0.58,0.030526
"""HMMM""",0.663265,0.65,0.65,0.013265
"""LLL""",0.427083,0.41,0.41,0.017083
…,…,…,…,…
"""HHHH""",0.71134,0.69,0.69,0.02134
"""HLL""",0.479592,0.47,0.47,0.009592
"""L (Baseline)""",0.372781,0.315,0.335,0.057781


-> Using (null) is the best, as unparsed values increase your score... should not be used

-> Using wrong is the worst, could possibly be used to be fair, as it is "not correct"

-> Papers and Benchmarks often use "random", which increases the values artificially and introduces noise.. I do not like it, but to be fair one should use it.

---

-> But in all of out cases, even absolute deviation is actually pretty low. (it gets mitigated in group decisions, as unparsable values are ignored in aggregation)

# Agreeableness -> Number of Cases where they reach conslusion

In [15]:
unanimity = (
    mad_analysis.with_columns(
        pl.col(AnalysisColumn.parsed_individual_answers_after.value)
        .list.n_unique()
        .eq(1)
        .alias("End unanimity"),
        pl.col(AnalysisColumn.parsed_individual_answers_before.value)
        .list.n_unique()
        .eq(1)
        .alias("Start unanimity"),
    )
    .group_by("group_constellation")
    .agg(
        pl.col("End unanimity").mean(),
        pl.col("Start unanimity").mean(),
        pl.when(pl.col("Start unanimity"))
        .then(None)
        .otherwise(pl.col("End unanimity"))
        .mean()
        .alias("End Unanimity | not Start Unanimity"),
        pl.when(pl.col("Start unanimity"))
        .then(pl.col("End unanimity"))
        .otherwise(None)
        .mean()
        .alias("End Unanimity | Start Unanimity"),
        accuracy=pl.col("is_correct").mean(),
    )
)

unanimity

group_constellation,End unanimity,Start unanimity,End Unanimity | not Start Unanimity,End Unanimity | Start Unanimity,accuracy
str,f64,f64,f64,f64,f64
"""HLLM""",0.74,0.2,0.7,0.9,0.58
"""HHL""",0.75,0.3,0.685714,0.9,0.65
"""HH""",0.96,0.79,0.904762,0.974684,0.76
"""HLLL""",0.59,0.16,0.535714,0.875,0.43
"""L""",1.0,1.0,null,1.0,0.25
…,…,…,…,…,…
"""HHLM""",0.83,0.29,0.802817,0.896552,0.69
"""MM""",0.93,0.88,0.916667,0.931818,0.61
"""HM""",0.88,0.74,0.846154,0.891892,0.61


-> In Human Groups (see Group Problem Solving) there is the tendency that the smarter the group, the more it is a "Truth Supported" Decision Scheme, the "dumber" the group, the more it is "proportional"

### Correlation for groups (ignoring 1 member, as it is always 1)

In [16]:
print("Pearson Correlation:")
print(
    unanimity.filter(pl.col("group_constellation").str.len_chars() > 1)
    .select(pl.exclude("group_constellation"))
    .corr()
)

print("Spearman Rank Correlation:")

print(
    unanimity.filter(pl.col("group_constellation").str.len_chars() > 1)
    .select(pl.exclude("group_constellation"))
    .with_columns(pl.all().rank())
    .corr()
)

Pearson Correlation:
shape: (5, 5)
┌───────────────┬─────────────────┬─────────────────────┬───────────────────────┬──────────┐
│ End unanimity ┆ Start unanimity ┆ End Unanimity | not ┆ End Unanimity | Start ┆ accuracy │
│ ---           ┆ ---             ┆ Start Unan…         ┆ Unanimit…             ┆ ---      │
│ f64           ┆ f64             ┆ ---                 ┆ ---                   ┆ f64      │
│               ┆                 ┆ f64                 ┆ f64                   ┆          │
╞═══════════════╪═════════════════╪═════════════════════╪═══════════════════════╪══════════╡
│ 1.0           ┆ 0.827728        ┆ 0.927171            ┆ 0.527695              ┆ 0.420484 │
│ 0.827728      ┆ 1.0             ┆ 0.716343            ┆ 0.476741              ┆ 0.335886 │
│ 0.927171      ┆ 0.716343        ┆ 1.0                 ┆ 0.252307              ┆ 0.416728 │
│ 0.527695      ┆ 0.476741        ┆ 0.252307            ┆ 1.0                   ┆ 0.191466 │
│ 0.420484      ┆ 0.335886        ┆

-> significantly correlated

---
-> Accuracy Correlates with Unanimity
Of course this could be that if everyone is correct in the beginning, then if they that way, then the chance of being correct is higher,
But can we increase the accuracy by accepting when a group starts with one answer?

## Analysis: Is the group finding answers "together" ? (e.g. can it find answers out of wrong start)

In [17]:
correctness_influence = (
    mad_analysis.with_columns(
        correct_before=comparer(
            pl.col(AnalysisColumn.parsed_combined_answers_before.value),
            pl.col("answer_string"),
        )
    )
    .group_by("group_constellation")
    .agg(
        pl.when(pl.col("correct_before"))
        .then(pl.col("is_correct"))
        .otherwise(None)
        .mean()
        .alias("Correct | Correct in Beginning"),
        pl.when(pl.col("correct_before"))
        .then(None)
        .otherwise(pl.col("is_correct"))
        .mean()
        .alias("Correct | !Correct in Beginning"),
        accuracy=pl.col("is_correct").mean(),
    )
)

correctness_influence

group_constellation,Correct | Correct in Beginning,Correct | !Correct in Beginning,accuracy
str,f64,f64,f64
"""LLM""",0.860465,0.333333,0.56
"""HLLL""",0.813953,0.140351,0.43
"""MMMM""",0.901639,0.102564,0.59
"""HLM""",0.967213,0.205128,0.67
"""H""",0.944444,0.142857,0.72
…,…,…,…
"""LMM""",0.983051,0.121951,0.63
"""M""",0.842105,0.069767,0.51
"""HHHH""",0.957143,0.066667,0.69


While the difference in Correct | Correct in Beginning is negligible, the true power lies in changing the answer when They are not correct.

In [18]:
(
    correctness_influence.join(unanimity, how="left", on="group_constellation")
    .filter(pl.col("group_constellation").str.len_chars() > 1)
    .select(pl.exclude("group_constellation"))
    .with_columns(pl.all().rank())
    .corr()
)

Correct | Correct in Beginning,Correct | !Correct in Beginning,accuracy,End unanimity,Start unanimity,End Unanimity | not Start Unanimity,End Unanimity | Start Unanimity,accuracy_right
f64,f64,f64,f64,f64,f64,f64,f64
1.0,0.01958,0.714517,0.472904,0.351131,0.463742,0.419853,0.714517
0.01958,1.0,-0.12255,-0.351177,-0.391212,-0.245079,-0.149016,-0.12255
0.714517,-0.12255,1.0,0.493686,0.349566,0.502777,0.285166,1.0
0.472904,-0.351177,0.493686,1.0,0.8586,0.878381,0.618211,0.493686
0.351131,-0.391212,0.349566,0.8586,1.0,0.754214,0.455629,0.349566
0.463742,-0.245079,0.502777,0.878381,0.754214,1.0,0.297922,0.502777
0.419853,-0.149016,0.285166,0.618211,0.455629,0.297922,1.0,0.285166
0.714517,-0.12255,1.0,0.493686,0.349566,0.502777,0.285166,1.0


## Influence of Group Aggregation Protocol

In [19]:
group_reply_influence_table = pl.DataFrame(
    {
        "group_constellation": mad_analysis["group_constellation"]
        .unique()
        .extend(baseline_frame_with_group_constellation["group_constellation"].unique())
        .sort()
    }
)

for strategy in [MajorityVote(), SingularityVote()]:
    new_group_reply_agg = GroupReplyAggregator(strategy)
    group_reply_influence_table = group_reply_influence_table.join(
        apply_parsing_and_group_decision(
            mad_frame, parser, comparer, new_group_reply_agg
        )
        .with_columns(make_group_constellation())
        .group_by("group_constellation")
        .agg(accuracy=pl.col("is_correct").mean())
        .select(
            "group_constellation",
            pl.col("accuracy").alias(f"Accuracy ({strategy.__class__.__name__})"),
        ),
        on="group_constellation",
        how="inner",
    )

group_reply_influence_table = group_reply_influence_table.with_columns(
    deviation=(
        pl.max_horizontal(pl.exclude("group_constellation"))
        - pl.min_horizontal(pl.exclude("group_constellation"))
    ).alias("range")
)

group_reply_influence_table

group_constellation,Accuracy (MajorityVote),Accuracy (SingularityVote),deviation
str,f64,f64,f64
"""HLLM""",0.58,0.54,0.04
"""HHLL""",0.51,0.42,0.09
"""HHMM""",0.6,0.55,0.05
"""LL""",0.31,0.31,0.0
"""HLM""",0.67,0.6,0.07
…,…,…,…
"""HH""",0.76,0.74,0.02
"""LM""",0.52,0.5,0.02
"""HHHM""",0.64,0.62,0.02


In [20]:
group_reply_influence_table.drop("group_constellation").with_columns(
    pl.all().rank()
).corr()

Accuracy (MajorityVote),Accuracy (SingularityVote),deviation
f64,f64,f64
1.0,0.93389,0.050611
0.93389,1.0,-0.225041
0.050611,-0.225041,1.0


-> Slightly Negative Correlation between Accuracy and the deviation, meaning the better the more MajorityVote == SingularityVote -> Same argument as before

## Inter-Model Correctness Correlation (Aka answer diversity)

In [21]:
mad_analysis

shape: (3_400, 23)
┌───────┬────────┬────────────┬────────────┬───┬────────────┬────────────┬────────────┬────────────┐
│ id    ┆ run_id ┆ question_i ┆ phoenix_sp ┆ … ┆ ___parsed_ ┆ ___parsed_ ┆ is_correct ┆ group_cons │
│ ---   ┆ ---    ┆ d          ┆ an_id      ┆   ┆ combined_a ┆ combined_a ┆ ---        ┆ tellation  │
│ i64   ┆ i64    ┆ ---        ┆ ---        ┆   ┆ nswers_bef ┆ nswers_aft ┆ bool       ┆ ---        │
│       ┆        ┆ i64        ┆ str        ┆   ┆ …          ┆ …          ┆            ┆ str        │
│       ┆        ┆            ┆            ┆   ┆ ---        ┆ ---        ┆            ┆            │
│       ┆        ┆            ┆            ┆   ┆ str        ┆ str        ┆            ┆            │
╞═══════╪════════╪════════════╪════════════╪═══╪════════════╪════════════╪════════════╪════════════╡
│ 41560 ┆ 95     ┆ 87         ┆ e12c8c5200 ┆ … ┆ J          ┆ ___all_vot ┆ false      ┆ M          │
│       ┆        ┆            ┆ f7c5d2     ┆   ┆            ┆ es_invalid ┆            ┆            │
│       ┆        ┆            ┆            ┆   ┆            ┆ ___        ┆            ┆            │
│ 41561 ┆ 95     ┆ 14         ┆ 22438e7201 ┆ … ┆ E          ┆ E          ┆ true       ┆ M          │
│       ┆        ┆            ┆ aefc6c     ┆   ┆            ┆            ┆            ┆            │
│ 41562 ┆ 95     ┆ 18         ┆ b02dad9afc ┆ … ┆ H          ┆ A          ┆ false      ┆ M          │
│       ┆        ┆            ┆ 2bc8d2     ┆   ┆            ┆            ┆            ┆            │
│ 41563 ┆ 95     ┆ 16         ┆ 0099efe53f ┆ … ┆ C          ┆ C          ┆ true       ┆ M          │
│       ┆        ┆            ┆ fb0583     ┆   ┆            ┆            ┆            ┆            │
│ 41564 ┆ 95     ┆ 20         ┆ f99c72e075 ┆ … ┆ D          ┆ D          ┆ true       ┆ M          │
│       ┆        ┆            ┆ 9ccc0c     ┆   ┆            ┆            ┆            ┆            │
│ …     ┆ …      ┆ …          ┆ …          ┆ … ┆ …          ┆ …          ┆ …          ┆ …          │
│ 72895 ┆ 403    ┆ 48         ┆ 6c845f951e ┆ … ┆ ___differe ┆ H          ┆ false      ┆ HLM        │
│       ┆        ┆            ┆ a5375c     ┆   ┆ nt_votes__ ┆            ┆            ┆            │
│       ┆        ┆            ┆            ┆   ┆ _          ┆            ┆            ┆            │
│ 72896 ┆ 403    ┆ 82         ┆ 9867ef8ff8 ┆ … ┆ D          ┆ D          ┆ false      ┆ HLM        │
│       ┆        ┆            ┆ d78f33     ┆   ┆            ┆            ┆            ┆            │
│ 72897 ┆ 403    ┆ 81         ┆ 4997f19a88 ┆ … ┆ D          ┆ C          ┆ false      ┆ HLM        │
│       ┆        ┆            ┆ dd2053     ┆   ┆            ┆            ┆            ┆            │
│ 72898 ┆ 403    ┆ 49         ┆ 498f036b8b ┆ … ┆ E          ┆ E          ┆ false      ┆ HLM        │
│       ┆        ┆            ┆ ebea34     ┆   ┆            ┆            ┆            ┆            │
│ 72899 ┆ 403    ┆ 44         ┆ 1b051b8346 ┆ … ┆ ___differe ┆ F          ┆ true       ┆ HLM        │
│       ┆        ┆            ┆ ce175e     ┆   ┆ nt_votes__ ┆            ┆            ┆            │
│       ┆        ┆            ┆            ┆   ┆ _          ┆            ┆            ┆            │
└───────┴────────┴────────────┴────────────┴───┴────────────┴────────────┴────────────┴────────────┘

#### For Individual answers

In [22]:
individual_models_answer_per_question = (
    (
        baseline_frame.with_columns(
            parser(pl.col("final_answer")).alias(AnalysisColumn.parsed_answer.value)
        )
        .with_columns(
            pl.col("model_name").replace(MODEL_NAME_TO_LETTER_MAPPING),
            is_correct=comparer(
                pl.col(AnalysisColumn.parsed_answer.value), pl.col("answer_string")
            ),
        )
        .pivot(
            values="is_correct",
            index="question_id",
            on="model_name",
            aggregate_function="mean",
        )
    )
    .sort("question_id")
    .select("question_id", "L", "M", "H")
    .with_columns(pl.exclude("question_id").cast(bool))
)

individual_models_answer_per_question

question_id,L,M,H
i64,bool,bool,bool
0,false,true,true
1,true,true,true
2,true,true,true
3,true,true,true
4,true,true,true
…,…,…,…
95,false,false,false
96,false,false,false
97,false,true,true


In [23]:
individual_models_answer_per_question.drop("question_id").corr()

L,M,H
f64,f64,f64
1.0,0.236454,0.20895
0.236454,1.0,0.505945
0.20895,0.505945,1.0


--> Actually surprisingly different

In [24]:
df = pl.concat(
    [
        individual_models_answer_per_question.group_by(col)
        .agg(pl.all().exclude(col, "question_id").mean())
        .with_columns(
            pl.when(pl.col(col))
            .then(pl.lit(col + "_correct"))
            .otherwise(pl.lit(col + "_incorrect"))
            .alias("Given"),
            pl.when(pl.col(col)).then(pl.lit(1.0)).otherwise(pl.lit(0.0)).alias(col),
        )
        for col in individual_models_answer_per_question.columns
        if col != "question_id"
    ],
    how="diagonal",
)

df_single = df.select("Given", "L", "M", "H")

df_single

Given,L,M,H
str,f64,f64,f64
"""L_incorrect""",0.0,0.595238,0.666667
"""L_correct""",1.0,0.810345,0.844828
"""M_incorrect""",0.392857,0.0,0.428571
"""M_correct""",0.652778,1.0,0.902778
"""H_correct""",0.636364,0.844156,1.0
"""H_incorrect""",0.391304,0.304348,0.0


-> They are not completely overlapping in the single answer case.

This means could be some diversity effect going on

---

In [25]:
individual_groups_answer_per_question = (
    (
        mad_analysis.filter(pl.col("group_constellation").str.len_chars() == 1).pivot(
            values="is_correct",
            index="question_id",
            on="group_constellation",
            aggregate_function="mean",
        )
    )
    .sort("question_id")
    .with_columns(pl.exclude("question_id").cast(bool))
)

df_single_mad = pl.concat(
    [
        individual_groups_answer_per_question.group_by(col)
        .agg(pl.all().exclude(col, "question_id").mean())
        .with_columns(
            pl.when(pl.col(col))
            .then(pl.lit(col + "_correct"))
            .otherwise(pl.lit(col + "_incorrect"))
            .alias("Given"),
            pl.when(pl.col(col)).then(pl.lit(1.0)).otherwise(pl.lit(0.0)).alias(col),
        )
        for col in individual_models_answer_per_question.columns
        if col != "question_id"
    ],
    how="diagonal",
).select("Given", "L", "M", "H")

df_single_mad

Given,L,M,H
str,f64,f64,f64
"""L_correct""",1.0,0.88,0.92
"""L_incorrect""",0.0,0.386667,0.653333
"""M_incorrect""",0.061224,0.0,0.510204
"""M_correct""",0.431373,1.0,0.921569
"""H_incorrect""",0.071429,0.142857,0.0
"""H_correct""",0.319444,0.652778,1.0


In [26]:
combined_ind = individual_models_answer_per_question.join(
    individual_groups_answer_per_question, on="question_id", suffix="_mad"
)
pl.concat(
    [
        combined_ind.group_by(col)
        .agg(pl.all().exclude(col, "question_id").mean())
        .with_columns(
            pl.when(pl.col(col))
            .then(pl.lit(col + "_correct"))
            .otherwise(pl.lit(col + "_incorrect"))
            .alias("Given"),
            pl.when(pl.col(col)).then(pl.lit(1.0)).otherwise(pl.lit(0.0)).alias(col),
        )
        for col in combined_ind.columns
        if col != "question_id"
    ],
    how="diagonal",
).select("Given", pl.exclude("Given"))

Given,L,M,H,M_mad,L_mad,H_mad
str,f64,f64,f64,f64,f64,f64
"""L_correct""",1.0,0.810345,0.844828,0.62069,0.396552,0.775862
"""L_incorrect""",0.0,0.595238,0.666667,0.357143,0.047619,0.642857
"""M_incorrect""",0.392857,0.0,0.428571,0.035714,0.035714,0.321429
"""M_correct""",0.652778,1.0,0.902778,0.694444,0.333333,0.875
"""H_incorrect""",0.391304,0.304348,0.0,0.043478,0.043478,0.130435
…,…,…,…,…,…,…
"""M_mad_correct""",0.705882,0.980392,0.980392,1.0,0.431373,0.921569
"""L_mad_correct""",0.92,0.96,0.96,0.88,1.0,0.92
"""L_mad_incorrect""",0.466667,0.64,0.706667,0.386667,0.0,0.653333


> See Obsidian

#### Is the "nobody is right case" in HHH actually unanimous?

In [27]:
print("Answers where everyone was wrong:")

everyone_wrong = (
    mad_analysis.filter(pl.col("group_constellation") == "HHH")
    .explode(AnalysisColumn.parsed_individual_answers_before.value)
    .with_columns(
        is_individually_correct=comparer(
            pl.col(AnalysisColumn.parsed_individual_answers_before.value),
            pl.col("answer_string"),
        )
    )
    .group_by("question_id")
    .agg(
        number_wrong=pl.len(),
        answers=pl.col(AnalysisColumn.parsed_individual_answers_before.value).implode(),
        is_individually_correct=pl.col("is_individually_correct").implode(),
    )
    .filter(pl.col("is_individually_correct").list.eval(pl.element().not_()).list.all())
    .drop("number_wrong", "is_individually_correct")
)

print(everyone_wrong)
unanimous = (
    everyone_wrong["answers"]
    .filter(everyone_wrong["answers"].list.n_unique() == 1)
    .count()
)
print(f"Of that unanimous: {unanimous}")
print("Contentious:")
print(everyone_wrong["answers"].filter(everyone_wrong["answers"].list.n_unique() != 1))

Answers where everyone was wrong:
shape: (23, 2)
┌─────────────┬─────────────────────────────────┐
│ question_id ┆ answers                         │
│ ---         ┆ ---                             │
│ i64         ┆ list[str]                       │
╞═════════════╪═════════════════════════════════╡
│ 71          ┆ ["A", "A", "A"]                 │
│ 84          ┆ ["C", "___not_parsable___", "_… │
│ 59          ┆ ["C", "C", "C"]                 │
│ 9           ┆ ["B", "___not_parsable___", "B… │
│ 11          ┆ ["E", "E", "E"]                 │
│ …           ┆ …                               │
│ 26          ┆ ["C", "C", "C"]                 │
│ 35          ┆ ["___not_parsable___", "___not… │
│ 82          ┆ ["D", "D", "D"]                 │
│ 43          ┆ ["D", "D", "D"]                 │
│ 3           ┆ ["___not_parsable___", "___not… │
└─────────────┴─────────────────────────────────┘
Of that unanimous: 15
Contentious:
shape: (8,)
Series: 'answers' [list[str]]
[
	["C", "___not_parsabl

C# Intra-Group Correctness Correlation (Aka group diversity)

C# Intra-Group Correctness Correlation (Aka group diversity)